# 04. Web Mapping (Part 2) — Web Map Visualization

**Mapping Systems · Assignment: Web Map Visualization**

This notebook extends last week's HAR web map assignment with a proper backend
(Supabase + PostGIS) and **data-driven styling**. Run the cells top to bottom; the
Supabase steps require you to click around in the Supabase dashboard (that part can't
be automated from here), but every file this project needs is generated for you.

**Variables visualized:**
- `SeatingChoice` (categorical: sidewalk / roadway / both) → circle **color**
- distance from your clicked point (continuous) → circle **size** + **opacity**
- a derived compliance flag (binary) → circle **stroke**


## Step 0 — Setup

Create the project folders before writing any files into them.


In [9]:
from pathlib import Path

SUPABASE_DIR = Path("supabase")
WEBMAP_DIR = Path("web_map")

for d in (SUPABASE_DIR, WEBMAP_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Ready:", [str(d.resolve()) for d in (SUPABASE_DIR, WEBMAP_DIR)])


Ready: ['/Users/jingyuanyang/Desktop/mscdp summer/mapping system/Assignments  Web Map Visualization/supabase', '/Users/jingyuanyang/Desktop/mscdp summer/mapping system/Assignments  Web Map Visualization/web_map']


## Step 1 — Set up the Supabase backend

1. Create a free project at [supabase.com](https://supabase.com).
2. Enable PostGIS: **Database → Extensions → postgis** (install into a schema named
   `gis`).
3. Download the CSV from NYC Open Data's [Open Restaurants
   Inspections](https://data.cityofnewyork.us/Transportation/Open-Restaurants-Inspections/4dx7-axux)
   dataset.
4. In **Table Editor**, create a new table by importing that CSV, name it
   `open-restaurant-inspections`, and set `RestaurantInspectionID` as the primary key.
5. Open the **SQL Editor** and run the script written by the next cell, top to bottom.

It adds a PostGIS geometry column populated from `Latitude`/`Longitude`, a spatial
index, a public read policy, and `find_nearest_n_restaurants()` — the function `map.js`
calls on every map click. It returns `SeatingChoice` plus the two compliance columns
(`IsSidewayCompliant`, `IsRoadwayCompliant`) alongside the distance of each result from
the clicked point.


In [10]:
from pathlib import Path

_target = Path('supabase/setup.sql')
_target.parent.mkdir(parents=True, exist_ok=True)
_target.write_text(r"""-- =====================================================================
-- Supabase / PostGIS setup for the "open-restaurant-inspections" table
-- -----------------------------------------------------------------------
-- Run each numbered block, in order, in the Supabase SQL Editor.
-- Assumes:
--   1. You've already created a Supabase project and enabled the
--      PostGIS extension into a schema named `gis`
--      (Database > Extensions > postgis).
--   2. You've imported the Open Restaurants Inspections CSV
--      (https://data.cityofnewyork.us/Transportation/Open-Restaurants-Inspections/4dx7-axux)
--      into a table named public."open-restaurant-inspections", with
--      "RestaurantInspectionID" set as the primary key.
--
-- Real columns in that CSV (confirmed against the live dataset) include:
--   RestaurantInspectionID, RestaurantName, SeatingChoice,
--   LegalBusinessName, BusinessAddress, IsSidewayCompliant,
--   IsRoadwayCompliant, InspectedOn, Borough, Latitude, Longitude, ...
--
-- SeatingChoice is our chosen data-driven-styling variable: it's a
-- categorical (nominal) field with values "sidewalk", "roadway", and
-- "both", describing what kind of outdoor seating structure the
-- restaurant was inspected for.
-- =====================================================================


-- 1. Add a geography column and populate it from Latitude / Longitude
-- -----------------------------------------------------------------------
ALTER TABLE public."open-restaurant-inspections"
ADD COLUMN geometry gis.geography(POINT, 4326);

CREATE INDEX open_restaurant_inspections_geometry_idx
ON public."open-restaurant-inspections"
USING gist (geometry);

-- Some rows have blank/invalid coordinates — only update the ones that
-- parse cleanly as numbers.
UPDATE public."open-restaurant-inspections"
SET geometry = gis.ST_SetSRID(
    gis.st_makepoint("Longitude"::double precision, "Latitude"::double precision),
    4326
)
WHERE "Longitude" ~ '^[+-]?[0-9]+(\.[0-9]+)?$'
  AND "Latitude"  ~ '^[+-]?[0-9]+(\.[0-9]+)?$';


-- 2. Grant read access so the anon/browser client can query it
-- -----------------------------------------------------------------------
GRANT USAGE ON SCHEMA gis TO anon, authenticated;

ALTER TABLE public."open-restaurant-inspections" ENABLE ROW LEVEL SECURITY;

CREATE POLICY "Public read access"
ON public."open-restaurant-inspections"
FOR SELECT
USING (true);


-- 3. Spatial query function used by map.js on every map click
-- -----------------------------------------------------------------------
-- Given a clicked (lat, lon) and a search radius `n` in meters, returns
-- every inspected restaurant within that radius, ordered nearest-first,
-- along with the two variables the web map visualizes:
--   - seating_choice                (categorical: sidewalk / roadway / both)
--   - sidewalk/roadway compliance   (used to flag non-compliant setups)
--   - dist_meters                   (continuous: distance from the click)
CREATE OR REPLACE FUNCTION find_nearest_n_restaurants(
    lat double precision,
    lon double precision,
    n integer
)
RETURNS TABLE (
    restaurant_inspection_id public."open-restaurant-inspections"."RestaurantInspectionID"%TYPE,
    name                     public."open-restaurant-inspections"."RestaurantName"%TYPE,
    seating_choice           public."open-restaurant-inspections"."SeatingChoice"%TYPE,
    sidewalk_compliance      public."open-restaurant-inspections"."IsSidewayCompliant"%TYPE,
    roadway_compliance       public."open-restaurant-inspections"."IsRoadwayCompliant"%TYPE,
    lat                      double precision,
    long                     double precision,
    dist_meters              double precision
)
SET search_path = ''
LANGUAGE sql AS $$
    SELECT
        "RestaurantInspectionID",
        "RestaurantName",
        "SeatingChoice",
        "IsSidewayCompliant",
        "IsRoadwayCompliant",
        gis.st_y(geometry::gis.geometry) AS lat,
        gis.st_x(geometry::gis.geometry) AS long,
        gis.st_distance(geometry, gis.st_point(lon, lat)::gis.geography) AS dist_meters
    FROM public."open-restaurant-inspections"
    WHERE geometry IS NOT NULL
      AND gis.st_dwithin(geometry, gis.st_point(lon, lat)::gis.geography, n)
    ORDER BY geometry OPERATOR(gis.<->) gis.st_point(lon, lat)::gis.geography
    LIMIT 300;
$$;
""")
print(f"Wrote {_target}")


Wrote supabase/setup.sql


## Step 2 — The variable, and the styling rationale

*(Copy this section directly into your submission — it's the written description the
assignment asks for.)*

**Variable chosen: `SeatingChoice`.** It's a categorical (nominal) field — `sidewalk`,
`roadway`, or `both` — describing what kind of outdoor seating structure a restaurant
was inspected for. It's visualized as **circle color**, since hue is the right visual
variable for an unordered categorical field: no value is "more" than another, they're
just different kinds of setups.

Combined with that, every point also encodes **distance from the clicked point**
(`dist_meters`, continuous) through both **circle size** and **circle opacity**: closer
results render larger and more opaque, farther ones shrink and fade. Size and opacity
are ordered visual variables, appropriate for a ratio-scale quantity like distance, and
using two channels for the same variable keeps the pattern legible at a glance, before
anyone reads a single popup.

As a third, secondary encoding, each point's **stroke** flags whether it has *any*
compliance note on file (from `IsSidewayCompliant` / `IsRoadwayCompliant`) — a thin dark
stroke for a clean record, a thicker red stroke for one flagged for review. That's
binary, lower-priority information, so it's carried by the least visually dominant
channel (stroke) rather than competing with color or size.

Together: **color = what kind of seating, size + opacity = how far away, stroke = any
compliance flags** — three variables, encoded so they don't visually compete.


## Step 3 — Build the web map files

`index.html` / `style.css` / `map.js` — the same three-file structure from the previous
assignment, extended with the Supabase client, the radius-search interaction, and the
data-driven `paint` properties described above.


### `style.css`

In [3]:
from pathlib import Path

_target = Path('web_map/style.css')
_target.parent.mkdir(parents=True, exist_ok=True)
_target.write_text(r""":root {
  --bg: #0a0e13;
  --panel: rgba(17, 22, 29, 0.88);
  --panel-border: #232c36;
  --line: #1c242d;
  --text: #e7ecf1;
  --text-muted: #8996a3;
  --accent-sidewalk: #3fe1c4;
  --accent-roadway: #f2a65a;
  --accent-both: #b98af2;
  --accent-unknown: #7c8b99;
  --accent-flag: #f2685f;
  --font-display: "Space Grotesk", "Helvetica Neue", Arial, sans-serif;
  --font-mono: "IBM Plex Mono", "SFMono-Regular", Consolas, monospace;
}

* {
  box-sizing: border-box;
}

html,
body {
  margin: 0;
  padding: 0;
  height: 100%;
  background: var(--bg);
  color: var(--text);
  font-family: var(--font-display);
  overflow: hidden;
}

#map {
  position: absolute;
  inset: 0;
  width: 100%;
  height: 100%;
}

/* ---------- Floating control panel ---------- */

#panel {
  position: absolute;
  top: 16px;
  left: 16px;
  width: 310px;
  max-height: calc(100vh - 32px);
  overflow-y: auto;
  background: var(--panel);
  border: 1px solid var(--panel-border);
  border-radius: 12px;
  backdrop-filter: blur(14px) saturate(140%);
  -webkit-backdrop-filter: blur(14px) saturate(140%);
  box-shadow: 0 12px 40px rgba(0, 0, 0, 0.45);
  padding: 18px 18px 14px;
  z-index: 5;
}

.eyebrow {
  font-family: var(--font-mono);
  font-size: 10.5px;
  letter-spacing: 0.14em;
  text-transform: uppercase;
  color: var(--accent-sidewalk);
  margin: 0 0 4px;
}

#panel h1 {
  font-size: 18.5px;
  font-weight: 600;
  line-height: 1.25;
  margin: 0 0 6px;
  letter-spacing: -0.01em;
}

#panel .subtitle {
  font-size: 12.5px;
  color: var(--text-muted);
  line-height: 1.5;
  margin: 0 0 14px;
}

.stat-row {
  display: grid;
  grid-template-columns: repeat(3, 1fr);
  gap: 8px;
  margin-bottom: 14px;
}

.stat {
  border: 1px solid var(--line);
  border-radius: 8px;
  padding: 8px 6px;
  text-align: center;
  background: rgba(255, 255, 255, 0.02);
}

.stat .value {
  display: block;
  font-family: var(--font-mono);
  font-size: 17px;
  font-weight: 500;
  color: var(--text);
}

.stat .label {
  display: block;
  font-size: 9px;
  text-transform: uppercase;
  letter-spacing: 0.06em;
  color: var(--text-muted);
  margin-top: 2px;
}

.divider {
  height: 1px;
  background: var(--line);
  margin: 12px 0;
  border: none;
}

.section-label {
  font-size: 10.5px;
  text-transform: uppercase;
  letter-spacing: 0.1em;
  color: var(--text-muted);
  margin: 0 0 8px;
}

/* ---------- Radius slider ---------- */

.slider-row {
  margin-bottom: 4px;
}

.slider-row .slider-label {
  display: flex;
  justify-content: space-between;
  font-size: 12.5px;
  margin-bottom: 6px;
}

.slider-row .slider-label span:last-child {
  font-family: var(--font-mono);
  color: var(--accent-sidewalk);
}

#radius-slider {
  width: 100%;
  accent-color: var(--accent-sidewalk);
  cursor: pointer;
}

/* ---------- Legend ---------- */

.legend-item {
  display: flex;
  align-items: center;
  gap: 8px;
  font-size: 12.5px;
  padding: 3px 0;
  color: var(--text);
}

.swatch {
  width: 9px;
  height: 9px;
  border-radius: 50%;
  flex: 0 0 auto;
  box-shadow: 0 0 6px currentColor;
}

.swatch.sidewalk { background: var(--accent-sidewalk); color: var(--accent-sidewalk); }
.swatch.roadway { background: var(--accent-roadway); color: var(--accent-roadway); }
.swatch.both { background: var(--accent-both); color: var(--accent-both); }
.swatch.unknown { background: var(--accent-unknown); color: var(--accent-unknown); }

.size-legend {
  display: flex;
  align-items: flex-end;
  gap: 14px;
  padding: 6px 2px 2px;
}

.size-legend .size-item {
  display: flex;
  flex-direction: column;
  align-items: center;
  gap: 6px;
}

.size-legend .dot {
  border-radius: 50%;
  background: var(--accent-sidewalk);
  opacity: 0.85;
}

.size-legend .dot.near { width: 16px; height: 16px; }
.size-legend .dot.mid { width: 10px; height: 10px; opacity: 0.7; }
.size-legend .dot.far { width: 5px; height: 5px; opacity: 0.55; }

.size-legend .size-caption {
  font-size: 9.5px;
  color: var(--text-muted);
  text-transform: uppercase;
  letter-spacing: 0.06em;
}

.flag-legend {
  display: flex;
  align-items: center;
  gap: 8px;
  font-size: 12.5px;
  margin-top: 6px;
}

.flag-ring {
  width: 11px;
  height: 11px;
  border-radius: 50%;
  border: 2px solid var(--accent-flag);
  flex: 0 0 auto;
}

#panel .footnote {
  font-family: var(--font-mono);
  font-size: 10px;
  color: var(--text-muted);
  line-height: 1.5;
  margin-top: 12px;
}

#panel .error {
  font-family: var(--font-mono);
  font-size: 10.5px;
  color: var(--accent-flag);
  line-height: 1.5;
  margin-top: 10px;
}

#panel-toggle {
  position: absolute;
  top: 16px;
  left: 16px;
  z-index: 6;
  display: none;
  width: 38px;
  height: 38px;
  border-radius: 10px;
  border: 1px solid var(--panel-border);
  background: var(--panel);
  color: var(--text);
  font-family: var(--font-mono);
  font-size: 16px;
  cursor: pointer;
}

/* ---------- MapLibre control repositioning ---------- */

.maplibregl-ctrl-top-right {
  top: 16px;
}

/* ---------- Popups ---------- */

.maplibregl-popup-content {
  background: #10151c;
  color: var(--text);
  border: 1px solid var(--panel-border);
  border-radius: 10px;
  padding: 12px 14px;
  font-family: var(--font-mono);
  box-shadow: 0 10px 30px rgba(0, 0, 0, 0.5);
  max-width: 240px;
}

.maplibregl-popup-tip {
  border-top-color: #10151c !important;
  border-bottom-color: #10151c !important;
}

.popup-kind {
  display: inline-block;
  font-size: 9.5px;
  text-transform: uppercase;
  letter-spacing: 0.1em;
  padding: 2px 6px;
  border-radius: 4px;
  margin-bottom: 6px;
}

.popup-name {
  font-size: 13.5px;
  font-weight: 600;
  color: var(--text);
  margin-bottom: 4px;
  line-height: 1.3;
}

.popup-row {
  font-size: 11px;
  color: var(--text-muted);
  margin-bottom: 2px;
}

.popup-row b {
  color: var(--text);
  font-weight: 500;
}

.popup-flag {
  font-size: 10.5px;
  color: var(--accent-flag);
  margin-top: 6px;
}

/* ---------- Responsive ---------- */

@media (max-width: 720px) {
  #panel {
    width: min(88vw, 320px);
    transform: translateX(calc(-100% - 24px));
    transition: transform 0.2s ease;
  }
  #panel.open {
    transform: translateX(0);
  }
  #panel-toggle {
    display: flex;
    align-items: center;
    justify-content: center;
  }
}
""")
print(f"Wrote {_target}")


Wrote web_map/style.css


### `map.js`

In [4]:
from pathlib import Path

_target = Path('web_map/map.js')
_target.parent.mkdir(parents=True, exist_ok=True)
_target.write_text(r"""// =============================================================
// Web Map Visualization — 04. Web Mapping (Part 2)
// Extends the previous week's web map with a Supabase/PostGIS
// backend and data-driven styling.
//
// Variables visualized:
//   - SeatingChoice   (categorical: sidewalk / roadway / both)  -> circle color
//   - distance from the clicked point (continuous, meters)      -> circle size + opacity
//   - compliance flag (derived, binary)                         -> circle stroke
// =============================================================

// ---- Supabase configuration ------------------------------------------
// Replace these with your own project's values (Project Settings > API).
const SUPABASE_URL = "YOUR_SUPABASE_URL";
const SUPABASE_KEY = "YOUR_SUPABASE_ANON_KEY";

const { createClient } = window.supabase;
const supabaseClient = createClient(SUPABASE_URL, SUPABASE_KEY);

// ---- Map configuration -------------------------------------------------
const BASEMAP_STYLE = "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json";
const NYC_CENTER = [-73.98, 40.75];

const DEFAULT_RADIUS_M = 500;
const MIN_RADIUS_M = 100;
const MAX_RADIUS_M = 2000;

const SEATING_COLORS = {
  sidewalk: "#3fe1c4",
  roadway: "#f2a65a",
  both: "#b98af2",
  unknown: "#7c8b99",
};

let currentRadius = DEFAULT_RADIUS_M;
let lastClick = null; // { lng, lat }
let layersReady = false;

// ---- Map setup -----------------------------------------------------------
const map = new maplibregl.Map({
  container: "map",
  style: BASEMAP_STYLE,
  center: NYC_CENTER,
  zoom: 12,
});

map.addControl(new maplibregl.NavigationControl(), "top-right");
map.addControl(new maplibregl.ScaleControl({ maxWidth: 120, unit: "metric" }), "bottom-left");

const emptyFC = { type: "FeatureCollection", features: [] };

// ---- Helpers -------------------------------------------------------------

// Normalize the raw SeatingChoice string ("sidewalk" / "roadway" / "both",
// occasionally inconsistent case) into one of our four known categories.
function normalizeSeating(value) {
  const v = (value || "").toString().trim().toLowerCase();
  if (v === "sidewalk" || v === "roadway" || v === "both") return v;
  return "unknown";
}

// A restaurant is "flagged" if either compliance column has any non-empty
// note on file (e.g. "Non-Compliant", "Cease and Desist", "For HIQA Review").
// A blank value means no issue was recorded for that seating type.
function isFlagged(row) {
  const s = (row.sidewalk_compliance || "").toString().trim();
  const r = (row.roadway_compliance || "").toString().trim();
  return s.length > 0 || r.length > 0;
}

function complianceSummary(row) {
  const s = (row.sidewalk_compliance || "").toString().trim();
  const r = (row.roadway_compliance || "").toString().trim();
  if (!s && !r) return "No compliance issues on file";
  const parts = [];
  if (s) parts.push(`Sidewalk: ${s}`);
  if (r) parts.push(`Roadway: ${r}`);
  return parts.join(" &middot; ");
}

// Build a circle polygon (in degrees) approximating a `radiusMeters` ring
// around `center`, for the search-radius overlay.
function circlePolygon(center, radiusMeters, steps = 64) {
  const [lng, lat] = center;
  const latRad = (lat * Math.PI) / 180;
  const dLat = radiusMeters / 111320;
  const dLng = radiusMeters / (111320 * Math.cos(latRad));

  const coords = [];
  for (let i = 0; i <= steps; i++) {
    const theta = (i / steps) * 2 * Math.PI;
    coords.push([lng + dLng * Math.cos(theta), lat + dLat * Math.sin(theta)]);
  }
  return {
    type: "Feature",
    properties: {},
    geometry: { type: "Polygon", coordinates: [coords] },
  };
}

function escapeHtml(str) {
  const div = document.createElement("div");
  div.textContent = str == null ? "" : String(str);
  return div.innerHTML;
}

// Convert rows returned by the find_nearest_n_restaurants RPC into a
// GeoJSON FeatureCollection ready for the map. Each feature carries the
// query radius alongside dist_meters so the paint expressions can compute
// a normalized 0-1 "how close was this" ratio without needing to be
// rebuilt every time the radius slider changes.
function rowsToGeoJSON(rows, radiusMeters) {
  return {
    type: "FeatureCollection",
    features: rows.map((row) => {
      const seating = normalizeSeating(row.seating_choice);
      const flagged = isFlagged(row);
      return {
        type: "Feature",
        properties: {
          id: row.restaurant_inspection_id,
          name: row.name || "Unnamed restaurant",
          seating: seating,
          dist_meters: row.dist_meters,
          radius_at_query: radiusMeters,
          flagged: flagged,
          sidewalk_compliance: row.sidewalk_compliance,
          roadway_compliance: row.roadway_compliance,
        },
        geometry: { type: "Point", coordinates: [row.long, row.lat] },
      };
    }),
  };
}

// ---- Supabase query --------------------------------------------------------

async function queryNearby(lngLat, radiusMeters) {
  setStatus(`Searching within ${radiusMeters} m...`);

  const { data, error } = await supabaseClient.rpc("find_nearest_n_restaurants", {
    lat: lngLat.lat,
    lon: lngLat.lng,
    n: radiusMeters,
  });

  if (error) {
    console.error("Error fetching nearest restaurants:", error);
    setStatus(`Query failed: ${error.message}`, true);
    return;
  }

  const geojson = rowsToGeoJSON(data, radiusMeters);
  updateMapData(geojson, lngLat, radiusMeters);
  updateStats(geojson);
  setStatus(null);
}

// ---- Map layers ------------------------------------------------------------

function addLayers() {
  // Search-radius ring (drawn first, underneath everything else)
  map.addSource("search-radius", { type: "geojson", data: emptyFC });
  map.addLayer({
    id: "search-radius-fill",
    type: "fill",
    source: "search-radius",
    paint: { "fill-color": SEATING_COLORS.sidewalk, "fill-opacity": 0.05 },
  });
  map.addLayer({
    id: "search-radius-line",
    type: "line",
    source: "search-radius",
    paint: {
      "line-color": SEATING_COLORS.sidewalk,
      "line-width": 1.2,
      "line-dasharray": [2, 2],
      "line-opacity": 0.5,
    },
  });

  // Search origin marker
  map.addSource("search-origin", { type: "geojson", data: emptyFC });
  map.addLayer({
    id: "search-origin-point",
    type: "circle",
    source: "search-origin",
    paint: {
      "circle-radius": 5,
      "circle-color": "#ffffff",
      "circle-stroke-width": 2,
      "circle-stroke-color": "#0a0e13",
    },
  });

  // Nearby restaurants — the data-driven layer
  map.addSource("nearby-restaurants", { type: "geojson", data: emptyFC });
  map.addLayer({
    id: "nearby-restaurants-points",
    type: "circle",
    source: "nearby-restaurants",
    paint: {
      // Color encodes the categorical variable: seating type
      "circle-color": [
        "match",
        ["get", "seating"],
        "sidewalk",
        SEATING_COLORS.sidewalk,
        "roadway",
        SEATING_COLORS.roadway,
        "both",
        SEATING_COLORS.both,
        SEATING_COLORS.unknown,
      ],
      // Size AND opacity both encode the continuous variable: distance
      // from the clicked point, normalized against the current search
      // radius so the encoding stays meaningful as the radius changes.
      "circle-radius": [
        "interpolate",
        ["linear"],
        ["/", ["get", "dist_meters"], ["get", "radius_at_query"]],
        0,
        15,
        1,
        4,
      ],
      "circle-opacity": [
        "interpolate",
        ["linear"],
        ["/", ["get", "dist_meters"], ["get", "radius_at_query"]],
        0,
        0.95,
        1,
        0.45,
      ],
      // Stroke flags the derived compliance variable
      "circle-stroke-width": ["case", ["get", "flagged"], 2.5, 1],
      "circle-stroke-color": ["case", ["get", "flagged"], "#f2685f", "#0a0e13"],
    },
  });

  layersReady = true;
}

function updateMapData(geojson, origin, radiusMeters) {
  map.getSource("nearby-restaurants").setData(geojson);
  map.getSource("search-origin").setData({
    type: "FeatureCollection",
    features: [{ type: "Feature", properties: {}, geometry: { type: "Point", coordinates: [origin.lng, origin.lat] } }],
  });
  map.getSource("search-radius").setData({
    type: "FeatureCollection",
    features: [circlePolygon([origin.lng, origin.lat], radiusMeters)],
  });
}

// ---- Interactions ------------------------------------------------------------

map.on("load", () => {
  addLayers();

  map.on("click", (e) => {
    lastClick = e.lngLat;
    queryNearby(lastClick, currentRadius);
  });

  map.on("click", "nearby-restaurants-points", (e) => {
    // Stop this click from also re-triggering the whole-map handler above,
    // so clicking directly on a point opens its popup without moving the
    // search origin.
    e.preventDefault();

    const f = e.features[0];
    const p = f.properties;
    const color = SEATING_COLORS[p.seating] || SEATING_COLORS.unknown;

    new maplibregl.Popup({ closeButton: true, offset: 10 })
      .setLngLat(f.geometry.coordinates)
      .setHTML(
        `<div class="popup-kind" style="background:${color}22;color:${color}">${escapeHtml(p.seating)}</div>` +
          `<div class="popup-name">${escapeHtml(p.name)}</div>` +
          `<div class="popup-row"><b>${Math.round(p.dist_meters)} m</b> from your click</div>` +
          `<div class="popup-row">${complianceSummary(p)}</div>` +
          (p.flagged ? `<div class="popup-flag">&#9679; flagged for review</div>` : "")
      )
      .addTo(map);
  });

  map.on("mouseenter", "nearby-restaurants-points", () => (map.getCanvas().style.cursor = "pointer"));
  map.on("mouseleave", "nearby-restaurants-points", () => (map.getCanvas().style.cursor = ""));
});

// ---- UI: panel, slider, stats, status --------------------------------------

function updateStats(geojson) {
  const total = geojson.features.length;
  const flaggedCount = geojson.features.filter((f) => f.properties.flagged).length;
  const closest = total ? Math.round(Math.min(...geojson.features.map((f) => f.properties.dist_meters))) : "–";

  document.getElementById("stat-count").textContent = total;
  document.getElementById("stat-closest").textContent = total ? `${closest}m` : "–";
  document.getElementById("stat-flagged").textContent = flaggedCount;
}

function setStatus(message, isError = false) {
  const el = document.getElementById("status-line");
  if (!message) {
    el.textContent = "";
    el.className = "footnote";
    return;
  }
  el.textContent = message;
  el.className = isError ? "error" : "footnote";
}

const radiusSlider = document.getElementById("radius-slider");
const radiusValue = document.getElementById("radius-value");

radiusSlider.min = MIN_RADIUS_M;
radiusSlider.max = MAX_RADIUS_M;
radiusSlider.value = DEFAULT_RADIUS_M;
radiusValue.textContent = `${DEFAULT_RADIUS_M} m`;

radiusSlider.addEventListener("input", (e) => {
  currentRadius = Number(e.target.value);
  radiusValue.textContent = `${currentRadius} m`;
  if (lastClick && layersReady) {
    queryNearby(lastClick, currentRadius);
  }
});

document.getElementById("panel-toggle").addEventListener("click", () => {
  document.getElementById("panel").classList.toggle("open");
});
""")
print(f"Wrote {_target}")


Wrote web_map/map.js


### `index.html`

In [5]:
from pathlib import Path

_target = Path('web_map/index.html')
_target.parent.mkdir(parents=True, exist_ok=True)
_target.write_text(r"""<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>Open Restaurants — Seating &amp; Distance | Web Map Visualization</title>

    <link
      href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;600&family=IBM+Plex+Mono:wght@400;500;600&display=swap"
      rel="stylesheet"
    />

    <script src="https://unpkg.com/maplibre-gl@^5.6.1/dist/maplibre-gl.js"></script>
    <link href="https://unpkg.com/maplibre-gl@^5.6.1/dist/maplibre-gl.css" rel="stylesheet" />
    <script src="https://cdn.jsdelivr.net/npm/@supabase/supabase-js@2"></script>

    <link rel="stylesheet" href="style.css" />
  </head>
  <body>
    <div id="map"></div>

    <button id="panel-toggle" aria-label="Toggle info panel">☰</button>

    <aside id="panel">
      <p class="eyebrow">Mapping Systems · Web Map Visualization</p>
      <h1>Open Restaurants — Seating &amp; Distance</h1>
      <p class="subtitle">
        Click anywhere on the map to query the Open Restaurants Inspections dataset for
        nearby seating setups, live from Supabase/PostGIS.
      </p>

      <div class="stat-row">
        <div class="stat">
          <span class="value" id="stat-count">–</span>
          <span class="label">Results</span>
        </div>
        <div class="stat">
          <span class="value" id="stat-closest">–</span>
          <span class="label">Closest</span>
        </div>
        <div class="stat">
          <span class="value" id="stat-flagged">–</span>
          <span class="label">Flagged</span>
        </div>
      </div>

      <div class="slider-row">
        <div class="slider-label">
          <span>Search radius</span>
          <span id="radius-value">500 m</span>
        </div>
        <input type="range" id="radius-slider" />
      </div>

      <hr class="divider" />

      <p class="section-label">Seating type</p>
      <div class="legend-item"><span class="swatch sidewalk"></span>Sidewalk</div>
      <div class="legend-item"><span class="swatch roadway"></span>Roadway</div>
      <div class="legend-item"><span class="swatch both"></span>Both</div>
      <div class="legend-item"><span class="swatch unknown"></span>Unspecified</div>

      <p class="section-label" style="margin-top: 12px">Distance from click</p>
      <div class="size-legend">
        <div class="size-item">
          <div class="dot near"></div>
          <span class="size-caption">Near</span>
        </div>
        <div class="size-item">
          <div class="dot mid"></div>
          <span class="size-caption">Mid</span>
        </div>
        <div class="size-item">
          <div class="dot far"></div>
          <span class="size-caption">Far</span>
        </div>
      </div>

      <div class="flag-legend">
        <span class="flag-ring"></span>
        Flagged for review (sidewalk or roadway)
      </div>

      <p class="footnote" id="status-line"></p>
    </aside>

    <script src="map.js"></script>
  </body>
</html>
""")
print(f"Wrote {_target}")


Wrote web_map/index.html


## Step 4 — Add your Supabase credentials

Find these under **Project Settings → API** in your Supabase dashboard: the **Project
URL** and the **anon public** API key. Run the cell below and paste them in when
prompted — it writes them straight into `web_map/map.js`, replacing the placeholders.


In [6]:
SUPABASE_URL = "https://rrkmamzfwuytxvddyylj.supabase.co"
SUPABASE_KEY = "sb_publishable_7cX_COSDs4Vx_5ZUrJ27jg_eeL7iLKR"

map_js_path = WEBMAP_DIR / "map.js"
content = map_js_path.read_text()
content = content.replace("YOUR_SUPABASE_URL", SUPABASE_URL).replace("YOUR_SUPABASE_ANON_KEY", SUPABASE_KEY)
map_js_path.write_text(content)
print(f"Credentials written to {map_js_path}")

Credentials written to web_map/map.js


## Step 5 — Preview the web map inline

Starts a local server for `web_map/` and tries three ways to show it: auto-opening your
default browser, a plain clickable link (works even when the notebook's embedded
preview is sandboxed — this is common in VS Code), and an inline iframe.


In [11]:
import http.server
import socketserver
import threading
import sys
import webbrowser
import urllib.request
from IPython.display import IFrame, HTML, display

PORT = 8766

class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory=str(WEBMAP_DIR), **kwargs)
    def log_message(self, *args):
        pass  # keep notebook output clean

httpd = socketserver.TCPServer(("", PORT), QuietHandler)
server_thread = threading.Thread(target=httpd.serve_forever, daemon=True)
server_thread.start()

local_url = f"http://127.0.0.1:8766/index.html"

# Sanity check: can the KERNEL itself reach the server it just started?
try:
    with urllib.request.urlopen(local_url, timeout=3) as resp:
        print(f"Server is up and reachable from the kernel (HTTP {resp.status}).")
except Exception as e:
    print(f"WARNING: the kernel itself can't reach {local_url}: {e}")
    print("The server didn't start correctly — check for a port conflict above.")

if "google.colab" in sys.modules:
    # Colab runs the kernel on a remote VM, so the browser's 'localhost' is a
    # different machine entirely. Use Colab's own port-forwarding helper.
    from google.colab.output import serve_kernel_port_as_iframe
    serve_kernel_port_as_iframe(PORT, path="/index.html", height=650)
else:
    # VS Code's notebook-output webview applies a strict CSP that frequently
    # blocks <iframe> content served from http://localhost, even though the
    # server itself is running fine (a VS Code sandboxing quirk, not a bug
    # in the map). A plain hyperlink is NOT subject to that CSP, and opening
    # the system browser directly always works, so we do both as fallbacks.
    try:
        webbrowser.open(local_url)
        print(f"Opened {local_url} in your default browser.")
    except Exception:
        pass

    display(HTML(
        f'<p>If a browser tab didn\'t open automatically, or the iframe below '
        f'is blank/blocked (common in VS Code), click here instead: '
        f'<a href="{local_url}" target="_blank">{local_url}</a></p>'
    ))
    display(IFrame(src=local_url, width="100%", height=650))


Server is up and reachable from the kernel (HTTP 200).
Opened http://127.0.0.1:8766/index.html in your default browser.


### Stop the local server

Run this before re-running the server-start cell above, to avoid an "address already
in use" error, and again when you're done previewing.


In [8]:
httpd.shutdown()
httpd.server_close()
print("Server stopped.")


Server stopped.


## Submission checklist

- [ ] Screenshot of the map showing the data-driven styling (click somewhere with
      results, ideally showing a mix of seating types and at least one flagged point)
- [ ] The variable + styling description from Step 2 above
- [ ] A link to the `web_map/` folder (`index.html`, `map.js`, `style.css`) — runnable
      via Live Server or a local HTTP server from that link
